# 03 — Gemma-4 XAI · Inference & Evaluation (V14 - Chain-of-Thought / XAI)

In [1]:
import os, sys, subprocess
def run(cmd):
    print("Running:", cmd)
    subprocess.run(cmd, shell=True, check=True)
run('pip install bitsandbytes -q')
run('pip install "unsloth_zoo @ git+https://github.com/unslothai/unsloth-zoo.git" -q')
run('pip install "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git" -q')
print('Installation complete.')


Running: pip install bitsandbytes -q
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.8 MB/s eta 0:00:00
Running: pip install "unsloth_zoo @ git+https://github.com/unslothai/unsloth-zoo.git" -q
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 65.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 75.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 660.6/660.6 kB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 81.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3/199.3 kB 16.5 MB/s eta 0:00

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
s3fs 2026.2.0 requires fsspec==2026.2.0, but you have fsspec 2025.9.0 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2025.9.0 which is incompatible.


Running: pip install "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git" -q
Installation complete.


In [2]:
import torch, json, gc, glob, datetime
from tqdm import tqdm
from huggingface_hub import login
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template

os.environ['CUDA_VISIBLE_DEVICES'] = '0'
try:
    from kaggle_secrets import UserSecretsClient
    VaccineNLP_TOKEN = UserSecretsClient().get_secret('VaccineNLP_TOKEN')
    if VaccineNLP_TOKEN: login(token=VaccineNLP_TOKEN)
except: pass

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
HF_MODEL_NAME = 'quynhphuong1209/gemma-4-E4B-unsloth-vaccine-xai'
found = glob.glob('/kaggle/input/**/benchmark_test_set.jsonl', recursive=True)
TEST_PATH   = found[0] if found else '/kaggle/input/vaccinenlp-clean-data/03_processed/benchmark_test_set.jsonl'
# [CHANGE 3] Đổi tên file output sang v14_XAI để tránh ghi đè dữ liệu v13
TEMP_FILE   = '/kaggle/working/gemma_inference_progress_v14_XAI.jsonl'
print(f'Test data: {TEST_PATH}')
print(f'Output file: {TEMP_FILE}')


Test data: /kaggle/input/datasets/inhlqunhphng/vaccinenlp-clean-data/03_processed/benchmark_test_set.jsonl
Output file: /kaggle/working/gemma_inference_progress_v14_XAI.jsonl


In [4]:
model, tokenizer = FastModel.from_pretrained(
    model_name     = HF_MODEL_NAME,
    max_seq_length = 1024,
    load_in_4bit   = True,
)
FastModel.for_inference(model)
tokenizer = get_chat_template(tokenizer, chat_template='gemma-4')


==((====))==  Unsloth 2026.4.8: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/147M [00:00<?, ?B/s]

In [5]:
REV_MISINFO = {'không liên quan': 0, 'khong lien quan': 0, 'tin giả': 1, 'tin gia': 1, 'chính xác': 2, 'chinh xac': 2,
               'misinformation': 1, 'accurate': 2, 'not relevant': 0, 'no misinformation': 2,
               'không có': 0, 'khong co': 0}
REV_STANCE  = {'ủng hộ': 0, 'ung ho': 0, 'support': 0, 'pro-vaccine': 0,
               'phản đối': 1, 'phan doi': 1, 'oppose': 1, 'anti-vaccine': 1, 'anti vaccine': 1,
               'trung lập': 2, 'trung lap': 2, 'neutral': 2, 'hỏi': 2, 'hoi': 2,
               'không rõ': 3, 'khong ro': 3, 'unknown': 3}
REV_SENTIMENT = {'tiêu cực': 0, 'tieu cuc': 0, 'negative': 0,
                 'trung tính': 1, 'trung tinh': 1, 'neutral': 1, 'trung tính': 1,
                 'tích cực': 2, 'tich cuc': 2, 'positive': 2}

# [CHANGE 1] XAI Prompt: Bắt buộc mô hình sinh Chain-of-Thought TRƯỚC khi chốt nhãn
def build_prompt(text):
    raw_prompt = (
        f"You are an Explainable AI in Public Health. For the following text, you MUST first write a detailed reasoning paragraph in Vietnamese analyzing the medical truth, the speaker's stance, and their sentiment. "
        f"ONLY AFTER your reasoning is complete, you must provide the final labels on a new line EXACTLY in this format: 'Kết quả: [Misinformation label] | [Stance label] | [Sentiment label]'.\n\n"
        f"Văn bản: {text}"
    )
    return [{'role': 'user', 'content': [{'type': 'text', 'text': raw_prompt}]}]

# [CHANGE 4] Parser XAI: Tách gemma_reasoning (TRƯỚC 'Kết quả:') và labels (SAU 'Kết quả:')
def parse_output(response):
    m, st, se = 0, 2, 1  # Default fallback
    gemma_reasoning = ''
    label_str = response
    parse_ok = False

    # Tách reasoning và label tại marker 'Kết quả:'
    MARKER = 'kết quả:'
    low = response.lower()
    marker_pos = low.find(MARKER)

    if marker_pos != -1:
        # Có marker → trích xuất reasoning và label_str
        gemma_reasoning = response[:marker_pos].strip()
        label_str = response[marker_pos + len(MARKER):].strip()
        parse_ok = True
    else:
        # Không có marker → ghi nhận raw_response làm fallback reasoning
        gemma_reasoning = ''
        label_str = response
        parse_ok = False

    # Parse labels từ label_str bằng split('|')
    parts = [p.strip().lower() for p in label_str.split('|')] if '|' in label_str else [label_str.lower()]

    # MISINFO
    target_m = parts[0] if len(parts) > 0 else label_str.lower()
    for k, v in REV_MISINFO.items():
        if k in target_m:
            m = v; break

    # STANCE
    target_st = parts[1] if len(parts) > 1 else label_str.lower()
    for k, v in REV_STANCE.items():
        if k in target_st:
            st = v; break

    # SENTIMENT
    target_se = parts[2] if len(parts) > 2 else label_str.lower()
    for k, v in REV_SENTIMENT.items():
        if k in target_se:
            se = v; break

    return m, st, se, parse_ok, gemma_reasoning


In [6]:
with open(TEST_PATH, 'r', encoding='utf-8') as fh:
    test_data = [json.loads(l) for l in fh]
print(f'Loaded {len(test_data)} samples from benchmark test set.')

temp_fp = open(TEMP_FILE, 'a', encoding='utf-8')

for idx, row in enumerate(tqdm(test_data)):
    text = row.get('text_cleaned') or row.get('text')
    convo = build_prompt(text)
    inputs = tokenizer.apply_chat_template(
        convo, tokenize=True, add_generation_prompt=True, return_tensors='pt'
    ).to('cuda')

    with torch.no_grad():
        # [CHANGE 2] max_new_tokens=512 để có đủ không gian sinh chuỗi lý luận dài
        outputs = model.generate(
            input_ids=inputs,
            max_new_tokens=512,
            temperature=0.1,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    m, st, se, ok, gemma_reasoning = parse_output(response)

    rec = {
        'id':             row.get('id', idx),
        'true_m':         int(row['standardized_ids'][0]),
        'pred_m':         m,
        'true_st':        int(row['standardized_ids'][1]),
        'pred_st':        st,
        'true_se':        int(row['standardized_ids'][2]),
        'pred_se':        se,
        'parse_ok':       ok,
        'gemma_reasoning': gemma_reasoning,  # [NEW] Chuỗi lý luận XAI
        'raw_response':   response           # Raw backup
    }

    temp_fp.write(json.dumps(rec, ensure_ascii=False) + '\n')
    temp_fp.flush()

    # In mẫu debug cho 5 sample đầu
    if idx < 5:
        print(f'\n--- Sample {idx} ---')
        print(f'Reasoning: {gemma_reasoning[:200]}...')
        print(f'Parsed: {m} | {st} | {se} | parse_ok={ok}')

    del inputs, outputs; gc.collect(); torch.cuda.empty_cache()

temp_fp.close()
print('\n[V14-XAI] Inference complete. Progress saved to ' + TEMP_FILE)


Loaded 186 samples from benchmark test set.


  0%|          | 0/186 [00:00<?, ?it/s]The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



--- Sample 0 ---
Reasoning: **Phân tích:**

Văn bản này là một đoạn trích từ vlog cá nhân về trải nghiệm tiêm vaccine Moderna, được đăng tải trong bối cảnh dịch COVID-19 và các quy định giãn cách xã hội (Chỉ thị 16). Về mặt y kh...
Parsed: 1 | 0 | 2 | parse_ok=True


  1%|          | 2/186 [02:30<3:48:39, 74.56s/it]


--- Sample 1 ---
Reasoning: **Phân tích:**

Về mặt y tế, câu chuyện này không chứa bất kỳ thông tin y tế nào cần được đánh giá về tính đúng sai về mặt khoa học. Nó là một tường thuật về một sự cố an toàn giao thông và hành vi củ...
Parsed: 0 | 2 | 2 | parse_ok=True


...


--- Sample 3 ---
Reasoning: **Phân tích:**

Văn bản này là một câu hỏi ngắn gọn, mang tính chất giao tiếp trong bối cảnh y tế công cộng, cụ thể là về việc tiêm chủng. Về mặt y khoa, câu hỏi này hoàn toàn hợp lý và cần thiết để x...
Parsed: 0 | 2 | 1 | parse_ok=True


  3%|▎         | 5/186 [04:51<2:40:38, 53.25s/it]


--- Sample 4 ---
Reasoning: **Phân tích:**

Văn bản này là một lời chứng thực cá nhân (testimonial) về một phương pháp hoặc lối sống nào đó, được người nói cho là đã mang lại sự cải thiện rõ rệt về sức khỏe. Về mặt y khoa, vì vă...
Parsed: 1 | 0 | 2 | parse_ok=True


100%|██████████| 186/186 [2:44:39<00:00, 53.11s/it]


[V14-XAI] Inference complete. Progress saved to /kaggle/working/gemma_inference_progress_v14_XAI.jsonl
